# Preprocessing: indicator features, scaling, train/test split

Three steps, in the order that avoids data leakage:

1. **Train/test split** (done in previous in data wrangling)
2. **Indicator (dummy) features** for categorical variables
3. **Standardization** of numeric features (scaler fit on the training set only)


## 0. Configuration

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from pathlib import Path

DATA_PATH   = Path("../outputs")   
TARGET      = "fail"                 # pass/fail column (1 = fail, 0 = pass)
TIME_COL    = "timestamp"                    # name of a timestamp column, or None if rows are already time-ordered
TIME_ORDERED = True                   # True -> chronological split; False -> stratified random split
TIME_FEATURES = False                 # True -> add cyclical hour / day-of-week features from the timestamp
TEST_SIZE   = 0.25
SEED        = 42
TRAIN_DATA = 'secom_train_clean.csv'
TEST_DATA ='secom_test_clean.csv'

## Train Test split was done in data wrangling step

In [2]:
# df = pd.read_csv(RAW / "secom.data", sep=" ", header=None, na_values=["NaN"])
#TEST_FRACTION = 0.2
#split_at = int(len(df) * (1 - TEST_FRACTION))
#X_train = df.iloc[:split_at].copy()
#X_test  = df.iloc[split_at:].copy()

## 1. Load data

In [3]:
df_train = pd.read_csv(DATA_PATH / TRAIN_DATA)
df_test = pd.read_csv(DATA_PATH / TEST_DATA)
print(f'Train data shape:', df_train.shape)
print(f'Test data shape:', df_test.shape)
df_test.head()

Train data shape: (1253, 466)
Test data shape: (314, 466)


,timestamp,fail,S000,S001,S002,S003,S004,S006,S007,S008,...,S556_isna,S557_isna,S562_isna,S563_isna,S564_isna,S565_isna,S566_isna,S567_isna,S568_isna,S569_isna
0,2008-10-02 20:54:00,0,3028.08,2525.56,2173.2778,1116.2950,0.8525,103.8200,0.1237,1.3697,...,0,0,0,0,0,0,0,0,0,0
1,2008-10-02 21:32:00,1,2988.41,2598.77,2173.2778,1116.2950,0.8525,103.8200,0.1237,1.3858,...,0,0,0,0,0,0,0,0,0,0
2,2008-10-02 21:33:00,0,2970.86,2510.19,2236.1111,1546.5931,2.0300,90.4233,0.1224,1.3582,...,0,0,0,0,0,0,0,0,0,0
3,2008-10-02 22:23:00,0,2914.04,2487.10,2238.1444,1580.6951,1.0062,91.0489,0.1230,1.4778,...,0,0,0,0,0,0,0,0,0,0
4,2008-10-02 22:33:00,0,2889.78,2434.42,2211.4000,1511.7842,1.3004,97.4700,0.1237,1.4596,...,0,0,0,0,0,0,0,0,0,0


## 2. Step 1: indicator (dummy) features
Categorical columns (text, category, or boolean) are one-hot encoded. `drop_first=True` removes one redundant level per variable, which avoids collinearity in linear models. Identifier and timestamp columns are excluded.

In [4]:
# apply one-hot encode to train data
y_train = df_train[TARGET].astype(int)
X_train = df_train.drop(columns=[TARGET])

time_values = None
if TIME_COL is not None:
    time_values = pd.to_datetime(X_train[TIME_COL])
    X_train = X_train.drop(columns=[TIME_COL])           # don't one-hot encode or feed the raw timestamp to the model
    if TIME_FEATURES:
        hour, dow = time_values.dt.hour, time_values.dt.dayofweek
        X_train["hour_sin"], X_train["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
        X_train["dow_sin"],  X_train["dow_cos"]  = np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7)
        print("Added cyclical time features")

cat_cols = X_train.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
print("Categorical columns:", cat_cols if cat_cols else "none (all numeric: this step is a no-op)")

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
print("Shape after encoding:", X_train.shape)

new_dummies = [c for c in X_train.columns if any(c.startswith(f"{k}_") for k in cat_cols)]
X_train[new_dummies].head() if new_dummies else X_train.iloc[:3, :5]

Categorical columns: none (all numeric: this step is a no-op)
Shape after encoding: (1253, 464)


,S000,S001,S002,S003,S004
0,3030.93,2564.00,2187.7333,1411.1265,1.3602
1,3095.78,2465.14,2230.4222,1463.6606,0.8294
2,2932.61,2559.94,2186.4111,1698.0172,1.5102


In [5]:
# apply one-hot encode to test data
y_test = df_test[TARGET].astype(int)
X_test = df_test.drop(columns=[TARGET])

if TIME_COL is not None:
    time_values = pd.to_datetime(X_test[TIME_COL])
    X_test = X_test.drop(columns=[TIME_COL])           # don't one-hot encode or feed the raw timestamp to the model
    if TIME_FEATURES:
        hour, dow = time_values.dt.hour, time_values.dt.dayofweek
        X_test["hour_sin"], X_test["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
        X_test["dow_sin"],  X_test["dow_cos"]  = np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7)
        print("Added cyclical time features")

cat_cols = X_test.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
print("Categorical columns:", cat_cols if cat_cols else "none (all numeric: this step is a no-op)")

X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=int)
print("Shape after encoding:", X_test.shape)

new_dummies = [c for c in X_test.columns if any(c.startswith(f"{k}_") for k in cat_cols)]
X_test[new_dummies].head() if new_dummies else X_test.iloc[:3, :5]

Categorical columns: none (all numeric: this step is a no-op)
Shape after encoding: (314, 464)


,S000,S001,S002,S003,S004
0,3028.08,2525.56,2173.2778,1116.2950,0.8525
1,2988.41,2598.77,2173.2778,1116.2950,0.8525
2,2970.86,2510.19,2236.1111,1546.5931,2.0300


### 3. Step 2: standardize numeric features
`StandardScaler` rescales each feature to mean 0 and standard deviation 1: `z = (x - mean_train) / std_train`.
This matters for scale-sensitive models (SVM, kNN, regularized regression, neural networks) and is harmless for trees.
Missing values are median-imputed first, since the scaler cannot handle NaN.

**Both transformers are fit on the training set only**, then applied unchanged to the test set.
Indicator columns are left as 0/1 (not scaled).

In [6]:
dummy_cols = [c for c in X_train.columns if set(X_train[c].dropna().unique()) <= {0, 1}
              and any(c.startswith(f"{k}_") for k in cat_cols)]
num_cols = [c for c in X_train.columns if c not in dummy_cols]

imputer = SimpleImputer(strategy="median").fit(X_train[num_cols])
scaler  = StandardScaler().fit(imputer.transform(X_train[num_cols]))

def transform(X_part):
    out = X_part.copy()
    out[num_cols] = scaler.transform(imputer.transform(X_part[num_cols]))
    return out

X_train_s = transform(X_train)
X_test_s  = transform(X_test)
print(f"{len(num_cols)} numeric columns scaled, {len(dummy_cols)} indicator columns untouched")

464 numeric columns scaled, 0 indicator columns untouched


#### 4. Verification

In [7]:
chk = pd.DataFrame({
    "train mean (should be ~0)": X_train_s[num_cols].mean().describe()[["min", "max"]],
    "train std (should be ~1)":  X_train_s[num_cols].std(ddof=0).describe()[["min", "max"]],
    "test mean (not exactly 0)": X_test_s[num_cols].mean().describe()[["min", "max"]],
})
display(chk)

assert np.allclose(X_train_s[num_cols].mean(), 0, atol=1e-8)
assert np.allclose(X_train_s[num_cols].std(ddof=0), 1, atol=1e-8)
assert not X_train_s.isna().any().any() and not X_test_s.isna().any().any()
print("All checks passed. Test-set means differ from 0 by design: the scaler never saw test data.")

,train mean (should be ~0),train std (should be ~1),test mean (not exactly 0)
min,-8.590025e-14,1.0,-1.070337
max,1.889488e-14,1.0,1.261771


All checks passed. Test-set means differ from 0 by design: the scaler never saw test data.


## 5. Save outputs

In [8]:
pd.concat([X_train_s, y_train.rename(TARGET)], axis=1).to_csv(DATA_PATH / "train_processed.csv", index=False)
pd.concat([X_test_s,  y_test.rename(TARGET)],  axis=1).to_csv(DATA_PATH / "test_processed.csv",  index=False)
print("Wrote train_processed.csv and test_processed.csv")

Wrote train_processed.csv and test_processed.csv
